# Audio Classifier

Interaktivní notebook pro klasifikaci a vizualizaci audia pomocí ContentVec embeddingů a UMAP.

**Workflow:**
1. Konfigurace parametrů
2. Načtení datasetu (složky = kategorie)
3. Extrakce 768D embeddingů (ContentVec / HuBERT)
4. UMAP redukce na 3D
5. Vizualizace a analýza

In [ ]:
import sys
sys.path.insert(0, "src")

from audio_classifier import (
    AudioClassifierPipeline,
    AudioConfig,
    ModelConfig,
    UMAPConfig,
    VisualizationConfig,
    PipelineConfig,
)
from audio_classifier.analysis import (
    calculate_centroid_distances,
    analyze_clustering_quality,
    print_clustering_report,
)

DATA_DIR = "./data"
OUTPUT_DIR = "./output"

In [ ]:
config = PipelineConfig(
    audio=AudioConfig(
        silence_removal_enabled=True,
        silence_threshold_db=40,
        max_duration_seconds=240,
        chunking_enabled=False,
        # chunk_duration_seconds=5.0,
        # chunk_handling="discard",
    ),
    model=ModelConfig(
        pooling="mean",
        layer=-1,
    ),
    umap=UMAPConfig(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
    ),
    visualization=VisualizationConfig(
        output_dir=OUTPUT_DIR,
        marker_size=3,
        marker_opacity=0.8,
        auto_open=False,
    ),
)

In [ ]:
pipeline = AudioClassifierPipeline(config)

In [ ]:
pipeline.load_dataset(DATA_DIR)

## Alternativa: pouze UMAP + vizualizace z cache

Pokud už máš embeddingy v cache a chceš jen změnit UMAP parametry nebo vizualizaci, přeskoč buňky výše a spusť odtud.

In [ ]:
# Načtení embeddingů z cache (přeskočí extrakci)
# pipeline.load_embeddings_from_cache()
# pipeline.reduce_dimensions()

## Extrakce embeddingů

ContentVec model (~360 MB) se stáhne automaticky při prvním spuštění.

In [ ]:
pipeline.extract_embeddings()

## UMAP redukce dimenzí

Redukce z 768D na 3D pro vizualizaci.

In [ ]:
pipeline.reduce_dimensions()

## Výsledky

In [ ]:
df = pipeline.get_dataframe()
print(f"Celkem vzorků: {len(df)}")
print(f"\nPočet vzorků podle kategorie:")
print(df["label"].value_counts())
df.head(10)

## 3D Vizualizace

In [ ]:
fig_3d = pipeline.visualizer.plot_3d_scatter(df, title="Audio Embeddings - 3D")
fig_3d.show()

## Heatmapa vzdáleností mezi kategoriemi

In [ ]:
dist_matrix = calculate_centroid_distances(df)
fig_dist = pipeline.visualizer.plot_distance_heatmap(dist_matrix, title="Vzdálenosti mezi kategoriemi")
fig_dist.show()

## Clustering analýza

In [ ]:
coords = df[["x", "y", "z"]].values
labels = df["label"].tolist()
metrics = analyze_clustering_quality(coords, labels)
print_clustering_report(metrics)

In [ ]:
# Uložení embeddingů do .npz
pipeline.save_embeddings(f"{OUTPUT_DIR}/embeddings.npz")

# Uložení výsledků do CSV
df.to_csv(f"{OUTPUT_DIR}/results.csv", index=False)
print(f"Uloženo: {OUTPUT_DIR}/results.csv")

# Uložení HTML vizualizací
pipeline.visualize_and_save()

In [ ]:
# pipeline = AudioClassifierPipeline(config)
# df = pipeline.run(DATA_DIR)
# pipeline.visualize_and_save()